# Degree:
file_path = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

# Company:
file_path = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"


# read all Column, row random, no empty row, show 10 row only

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# READ ALL COLUMNS + RANDOM NON-EMPTY ROWS
# NO ...
# Shows loading while reading big CSV files
# Does NOT load whole file at once
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

CHUNKSIZE = 100_000

# How many random rows to print from each file
ROWS_TO_SHOW = 10

# False = row only needs at least 1 real value
# True  = every column in the row must have a value
STRICT_NO_EMPTY_CELLS = False

RANDOM_SEED = 42

# Print loading every N chunks
LOAD_PRINT_EVERY_CHUNKS = 1


# ============================================================
# PANDAS DISPLAY SETTINGS
# This helps, but the main fix is using .to_string()
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.min_rows", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def file_size_mb(path):
    return path.stat().st_size / (1024 * 1024)


def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"File not found:\n{path}")

    print("FOUND FILE:")
    print(path)
    print(f"File size: {file_size_mb(path):,.2f} MB")
    print()


def print_full_table(df):
    """
    Print full DataFrame with no ...
    """
    print(
        df.to_string(
            index=False,
            max_rows=None,
            max_cols=None,
            line_width=None
        )
    )


def print_all_columns(path, label):
    print("=" * 100)
    print(f"{label} FILE COLUMNS")
    print("=" * 100)

    print(f"Loading columns from {label} file...", flush=True)

    header = pd.read_csv(path, nrows=0)

    columns_table = pd.DataFrame({
        "column_number": range(1, len(header.columns) + 1),
        "column_name": header.columns
    })

    print(f"Total columns: {len(header.columns)}")
    print()

    # IMPORTANT:
    # This prints every column name.
    # No display().
    # No ...
    print_full_table(columns_table)

    print()
    print(f"Finished loading columns for {label}.")
    print()

    return list(header.columns)


def make_non_empty_mask(chunk, strict_no_empty_cells=False):
    """
    strict_no_empty_cells = False:
        keep rows where at least one column has a real value

    strict_no_empty_cells = True:
        keep rows where every column has a real value
    """

    text_check = chunk.astype("string").fillna("")
    non_empty_cells = text_check.apply(lambda col: col.str.strip().ne(""))

    if strict_no_empty_cells:
        return non_empty_cells.all(axis=1)
    else:
        return non_empty_cells.any(axis=1)


def random_non_empty_rows(
    path,
    label,
    rows_to_show=10,
    chunksize=100_000,
    strict_no_empty_cells=False,
    random_seed=42
):
    print("=" * 100)
    print(f"RANDOM NON-EMPTY ROWS FROM {label}")
    print("=" * 100)

    print(f"START LOADING {label} FILE...")
    print(f"Chunk size: {chunksize:,}")
    print(f"Random rows wanted: {rows_to_show}")
    print(f"Strict no empty cells: {strict_no_empty_cells}")
    print()

    rng = np.random.default_rng(random_seed)

    best_rows = []
    total_rows_read = 0
    total_non_empty_rows = 0
    chunk_number = 0

    for chunk in pd.read_csv(
        path,
        chunksize=chunksize,
        dtype=str,
        keep_default_na=False,
        low_memory=False
    ):
        chunk_number += 1
        rows_in_chunk = len(chunk)
        total_rows_read += rows_in_chunk

        mask = make_non_empty_mask(
            chunk,
            strict_no_empty_cells=strict_no_empty_cells
        )

        chunk = chunk.loc[mask].copy()
        non_empty_in_chunk = len(chunk)
        total_non_empty_rows += non_empty_in_chunk

        if not chunk.empty:
            # Original CSV row number:
            # +2 because CSV line 1 is header, data starts on line 2
            chunk.insert(0, "csv_row_number", chunk.index + 2)

            # Add random key so we can keep only random rows
            chunk["_random_key"] = rng.random(len(chunk))

            best_rows.append(chunk)

            combined = pd.concat(best_rows, ignore_index=True)

            # Keep only random smallest keys
            combined = combined.nsmallest(rows_to_show, "_random_key")

            best_rows = [combined]

        if chunk_number % LOAD_PRINT_EVERY_CHUNKS == 0:
            print(
                f"Loading {label}... "
                f"chunk {chunk_number:,} | "
                f"rows read: {total_rows_read:,} | "
                f"non-empty rows found: {total_non_empty_rows:,} | "
                f"random rows kept: {len(best_rows[0]) if best_rows else 0}",
                flush=True
            )

    print()
    print(f"FINISHED LOADING {label}")
    print(f"Total chunks read: {chunk_number:,}")
    print(f"Total rows read: {total_rows_read:,}")
    print(f"Total non-empty rows found: {total_non_empty_rows:,}")
    print()

    if not best_rows:
        print("No non-empty rows found.")
        return pd.DataFrame()

    result = best_rows[0].drop(columns=["_random_key"]).reset_index(drop=True)

    print(f"RANDOM NON-EMPTY ROWS RESULT FROM {label}:")
    print()

    # IMPORTANT:
    # This prints all columns in the random rows.
    # No display().
    # No ...
    print_full_table(result)

    print()

    return result


# ============================================================
# RUN DEGREE FILE
# ============================================================

print("=" * 100)
print("START DEGREE ANALYSIS")
print("=" * 100)

check_file(DEGREE_FILE)

degree_columns = print_all_columns(DEGREE_FILE, "DEGREE")

degree_random_rows = random_non_empty_rows(
    DEGREE_FILE,
    label="DEGREE",
    rows_to_show=ROWS_TO_SHOW,
    chunksize=CHUNKSIZE,
    strict_no_empty_cells=STRICT_NO_EMPTY_CELLS,
    random_seed=RANDOM_SEED
)


# ============================================================
# RUN COMPANY FILE
# ============================================================

print("=" * 100)
print("START COMPANY ANALYSIS")
print("=" * 100)

check_file(COMPANY_FILE)

company_columns = print_all_columns(COMPANY_FILE, "COMPANY")

company_random_rows = random_non_empty_rows(
    COMPANY_FILE,
    label="COMPANY",
    rows_to_show=ROWS_TO_SHOW,
    chunksize=CHUNKSIZE,
    strict_no_empty_cells=STRICT_NO_EMPTY_CELLS,
    random_seed=RANDOM_SEED
)


# ============================================================
# FINISH
# ============================================================

print()
print("=" * 100)
print("FINISH ANALYSIS")
print("=" * 100)
print("Degree total columns:", len(degree_columns))
print("Company total columns:", len(company_columns))
print("Degree random rows shown:", len(degree_random_rows))
print("Company random rows shown:", len(company_random_rows))
print("DONE")

# All Column Year Coverage Check
Prompt / Description:

Check every column in the Degree file and Company file. For each column, show the available year range and 3 random non-empty sample values. Output one table with: File, Column #, Year column, Available year range, Key column, Random row 1, Random row 2, Random row 3.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import random

try:
    from IPython.display import display
except ImportError:
    display = print

# ============================================================
# ALL COLUMNS YEAR RANGE + 3 RANDOM NON-EMPTY ROW VALUES
# One row per column
#
# Output:
# File | Column # | Year column | Available year range
# Key columns to keep | Random row 1 | Random row 2 | Random row 3
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

CHUNKSIZE = 100_000
RANDOM_SEED = 42
SAMPLES_PER_COLUMN = 3

OUTPUT_FILE = Path.home() / "Downloads/all_columns_year_range_random_rows.csv"

pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 250)

rng = random.Random(RANDOM_SEED)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file:\n{path}")
    print("FOUND:", path)


def get_columns(path):
    return list(pd.read_csv(path, nrows=0).columns)


def clean_year_series(s):
    return pd.to_numeric(s, errors="coerce")


def real_value_mask(s):
    """
    True = real non-empty value.
    False = NaN, blank, 'nan', 'none', 'null'.
    0 is treated as real value.
    """
    if pd.api.types.is_numeric_dtype(s):
        return s.notna()

    temp = (
        s.astype(str)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "NaN": np.nan,
            "NAN": np.nan,
            "none": np.nan,
            "None": np.nan,
            "NONE": np.nan,
            "null": np.nan,
            "Null": np.nan,
            "NULL": np.nan,
            "<NA>": np.nan,
            "NaT": np.nan
        })
    )

    return temp.notna()


def clean_sample_value(x):
    if pd.isna(x):
        return ""

    text = str(x).strip()

    if text.endswith(".0"):
        try:
            return str(int(float(text)))
        except:
            return text

    return text


def sample_from_chunk(values_df, max_samples=3):
    """
    Randomly choose up to 3 rows from this chunk for one column.
    values_df must have columns: year, value
    """
    if values_df.empty:
        return []

    n = min(max_samples, len(values_df))
    sampled = values_df.sample(n=n, random_state=rng.randint(1, 10_000_000))

    result = []
    for _, row in sampled.iterrows():
        y = row["year"]
        v = clean_sample_value(row["value"])

        if pd.isna(y):
            result.append(v)
        else:
            try:
                result.append(f"{int(y)}: {v}")
            except:
                result.append(v)

    return result


def final_three_samples(candidate_samples):
    """
    From all chunk samples, choose final 3 random examples.
    """
    candidate_samples = [x for x in candidate_samples if str(x).strip() != ""]

    if len(candidate_samples) == 0:
        return ["", "", ""]

    n = min(3, len(candidate_samples))
    chosen = rng.sample(candidate_samples, n)

    while len(chosen) < 3:
        chosen.append("")

    return chosen


# ------------------------------------------------------------
# Main function
# ------------------------------------------------------------
def analyze_all_columns(file_label, path, year_col="year"):
    check_file(path)

    columns = get_columns(path)

    if year_col not in columns:
        raise ValueError(f"Year column '{year_col}' not found in {path.name}")

    print("\n" + "=" * 120)
    print(f"LOADING FILE: {file_label}")
    print(f"File name: {path.name}")
    print(f"Total columns: {len(columns)}")
    print("=" * 120)

    # Track result for every column
    col_stats = {}

    for col in columns:
        col_stats[col] = {
            "from_year": None,
            "up_to_year": None,
            "candidate_samples": []
        }

    # Read whole file in chunks
    for chunk_number, chunk in enumerate(
        pd.read_csv(
            path,
            chunksize=CHUNKSIZE,
            low_memory=False
        ),
        start=1
    ):
        print(f"Loading {file_label} chunk {chunk_number:,} | rows: {len(chunk):,}")

        years = clean_year_series(chunk[year_col])
        valid_year_mask = years.notna()

        if valid_year_mask.sum() == 0:
            continue

        years_valid = years[valid_year_mask].astype(int)

        for col in columns:
            values = chunk.loc[valid_year_mask, col]

            mask = real_value_mask(values)

            if mask.sum() == 0:
                continue

            valid_years_for_col = years_valid.loc[mask.index[mask]]

            if len(valid_years_for_col) == 0:
                continue

            min_year = int(valid_years_for_col.min())
            max_year = int(valid_years_for_col.max())

            if col_stats[col]["from_year"] is None:
                col_stats[col]["from_year"] = min_year
            else:
                col_stats[col]["from_year"] = min(col_stats[col]["from_year"], min_year)

            if col_stats[col]["up_to_year"] is None:
                col_stats[col]["up_to_year"] = max_year
            else:
                col_stats[col]["up_to_year"] = max(col_stats[col]["up_to_year"], max_year)

            # Random examples from this column
            non_empty_values = values[mask]

            sample_df = pd.DataFrame({
                "year": valid_years_for_col.values,
                "value": non_empty_values.values
            })

            chunk_samples = sample_from_chunk(sample_df, max_samples=SAMPLES_PER_COLUMN)
            col_stats[col]["candidate_samples"].extend(chunk_samples)

    # Build output rows
    rows = []

    for col_num, col in enumerate(columns, start=1):
        from_year = col_stats[col]["from_year"]
        up_to_year = col_stats[col]["up_to_year"]

        if from_year is None or up_to_year is None:
            available_year_range = "No non-empty values found"
        else:
            available_year_range = f"{from_year}–{up_to_year}"

        s1, s2, s3 = final_three_samples(col_stats[col]["candidate_samples"])

        rows.append({
            "File": file_label,
            "Column #": col_num,
            "Year column": year_col,
            "Available year range": available_year_range,
            "Key columns to keep": col,
            "Random row 1": s1,
            "Random row 2": s2,
            "Random row 3": s3
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Run both files
# ------------------------------------------------------------
degree_table = analyze_all_columns(
    file_label="Degree file",
    path=DEGREE_FILE,
    year_col="year"
)

company_table = analyze_all_columns(
    file_label="Company file",
    path=COMPANY_FILE,
    year_col="year"
)

all_columns_table = pd.concat(
    [degree_table, company_table],
    ignore_index=True
)


# ------------------------------------------------------------
# Show final table
# ------------------------------------------------------------
print("\n" + "=" * 120)
print("ALL COLUMNS YEAR RANGE + RANDOM ROW VALUES")
print("=" * 120)

display(all_columns_table)

print("\nCOPY-FRIENDLY FULL TABLE:")
print(all_columns_table.to_string(index=False))


# ------------------------------------------------------------
# Save to Downloads
# ------------------------------------------------------------
all_columns_table.to_csv(OUTPUT_FILE, index=False)

print("\nSaved table to:")
print(OUTPUT_FILE)

# All Column Year Coverage Check - Short
Check every column in the Degree file and Company file. For each column, find the year column it uses and the available year range. If multiple columns have the same available year range, group them into one row. Output one table with: File, Year column, Available year range, Column names. Do not include Column #. Do not include random sample values.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# COLUMN YEAR RANGE SUMMARY
# Groups columns with the SAME available year range into ONE row
#
# Output table:
# File | Year column | Available year range | Column names
#
# No Column #
# No sample values
# No saving file
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

CHUNKSIZE = 100_000

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)


# ------------------------------------------------------------
# Check file exists
# ------------------------------------------------------------
def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"File not found:\n{path}")
    print("FOUND:", path)


# ------------------------------------------------------------
# Find year column automatically
# ------------------------------------------------------------
def find_year_column(columns):
    lower_map = {c.lower().strip(): c for c in columns}

    if "year" in lower_map:
        return lower_map["year"]

    possible_names = [
        "survey_year",
        "data_year",
        "academic_year",
        "report_year",
        "fiscal_year",
        "yr"
    ]

    for name in possible_names:
        if name in lower_map:
            return lower_map[name]

    year_like = [c for c in columns if "year" in c.lower()]
    if year_like:
        return year_like[0]

    raise ValueError("No year column found. Please manually set the year column name.")


# ------------------------------------------------------------
# Real non-empty value checker
# ------------------------------------------------------------
def non_empty_mask(series):
    mask = series.notna()

    if series.dtype == "object" or str(series.dtype).startswith("string"):
        text = series.astype("string").str.strip().str.lower()
        mask = mask & ~text.isin(["", "nan", "none", "null", "na", "n/a", "<na>"])

    return mask


# ------------------------------------------------------------
# Scan one file
# ------------------------------------------------------------
def scan_file_column_year_ranges(file_label, file_path):
    print("\n" + "=" * 100)
    print(f"SCANNING {file_label.upper()} FILE")
    print("=" * 100)

    check_file(file_path)

    columns = pd.read_csv(file_path, nrows=0).columns.tolist()
    year_col = find_year_column(columns)

    print("Year column used:", year_col)
    print("Total columns:", len(columns))

    stats = {
        col: {
            "min_year": np.inf,
            "max_year": -np.inf,
            "has_value": False
        }
        for col in columns
    }

    chunk_number = 0

    for chunk in pd.read_csv(file_path, chunksize=CHUNKSIZE, low_memory=False):
        chunk_number += 1
        print(f"Loading {file_label}: chunk {chunk_number:,}")

        years = pd.to_numeric(chunk[year_col], errors="coerce")
        valid_year = years.notna()

        for col in columns:
            mask = non_empty_mask(chunk[col]) & valid_year

            if mask.any():
                min_y = int(years[mask].min())
                max_y = int(years[mask].max())

                stats[col]["has_value"] = True
                stats[col]["min_year"] = min(stats[col]["min_year"], min_y)
                stats[col]["max_year"] = max(stats[col]["max_year"], max_y)

    rows = []

    for col, info in stats.items():
        if info["has_value"]:
            available_range = f"{int(info['min_year'])}-{int(info['max_year'])}"
        else:
            available_range = "No non-empty values"

        rows.append({
            "File": file_label,
            "Year column": year_col,
            "Available year range": available_range,
            "Column name": col
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Run Degree + Company
# ------------------------------------------------------------
degree_table = scan_file_column_year_ranges("Degree", DEGREE_FILE)
company_table = scan_file_column_year_ranges("Company", COMPANY_FILE)

all_columns_table = pd.concat([degree_table, company_table], ignore_index=True)


# ------------------------------------------------------------
# Group same year range together
# ------------------------------------------------------------
summary_table = (
    all_columns_table
    .groupby(["File", "Year column", "Available year range"], dropna=False)["Column name"]
    .apply(lambda x: ", ".join(x.astype(str)))
    .reset_index()
    .rename(columns={"Column name": "Column names"})
)


# ------------------------------------------------------------
# Sort by starting year
# ------------------------------------------------------------
def range_start(x):
    try:
        return int(str(x).split("-")[0])
    except:
        return 999999

summary_table["sort_year"] = summary_table["Available year range"].apply(range_start)

summary_table = (
    summary_table
    .sort_values(["File", "sort_year", "Available year range"])
    .drop(columns=["sort_year"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Print final table only
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("COLUMN YEAR RANGE SUMMARY")
print("=" * 100)

display(summary_table)